## Import packages

In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import ast, os, pickle, random, re
from joblib import Parallel, delayed
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor

import networkx as nx
from networkx.drawing.nx_pydot import to_pydot
import dowhy
from dowhy import gcm
from dowhy.gcm.auto import AssignmentQuality

In [ ]:
# Make it easier to read
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows",1000)
pd.set_option('display.max_columns',100)

## Initialize variables

In [ ]:
bootstrap = 100
run_label = "A"
folder_name = run_label+"_"+str(bootstrap)+"b"

## Load data

In [ ]:
edge_summary = pd.read_csv(f"summaries/{run_label}/causal_disc_edge_summary_b{bootstrap}.csv")
fit_summary = pd.read_csv(f"summaries/{run_label}/scm_fit_summary_b{bootstrap}.csv")
benchmark_summary = pd.read_csv(f"summaries/{run_label}/benchmark_comparison_b{bootstrap}.csv")
node_log_summary = pd.read_csv(f"summaries/{run_label}/scm_node_assignment_log_b{bootstrap}.csv")

In [ ]:
file_name_raw = 'raw_data_for_model_'+run_label+'.csv'
final_df_raw = pd.read_csv(file_name_raw)
file_name_std = 'std_data_for_model_'+run_label+'.csv'
final_df_std = pd.read_csv(file_name_std)

In [ ]:
dir = Path(f"graph_objects/{run_label}")
graphs = {}
scms = {}

# Example filename: pc_b75_0_35_refined.pkl
pattern_1 = re.compile(
    r"(?P<model>[a-zA-Z_]+)_b(?P<bootstrap>\d+)_(?P<threshold>\d+(?:_\d+)?|[0-9]+)_(?P<stage>initial|refined)\.pkl$"
)
pattern_2 = re.compile(
    r"(?P<model>[a-zA-Z_]+)_b(?P<bootstrap>\d+)_(?P<threshold>\d+(?:_\d+)?|[0-9]+)_(?P<stage>refined_scm)\.pkl$"
)

for file in dir.glob("*.pkl"):
    match_1 = pattern_1.match(file.name)
    match_2 = pattern_2.match(file.name)
    if not match_1 and not match_2:
        print(f"Skipping unrecognized file: {file.name}")
        continue
    
    match = match_1 if match_1 else match_2
    model = match.group("model")
    bootstrap = int(match.group("bootstrap"))
    threshold = float(match.group("threshold").replace("_", "."))
    stage = match.group("stage")
    
    with open(file, "rb") as f:
        if match_1:
            G = pickle.load(f)
            # Store as nested dictionary: graphs[model][threshold][stage]
            graphs.setdefault(model, {}).setdefault(threshold, {})[stage] = G
        else:
            scm = pickle.load(f)
            # Store as nested dictionary: graphs[model][threshold][stage]
            scms.setdefault(model, {}).setdefault(threshold, {})[stage] = scm
    

print("Loaded graphs for models:")
for model, thresholds in graphs.items():
    print(f"  {model}: thresholds = {list(thresholds.keys())}")
print("Loaded scms for models:")
for model, threhsolds in scms.items():
    print(f"  {model}: thresholds = {list(thresholds.keys())}")

## Initial analysis

### Edge summary

In [ ]:
rows = []

for model, thresholds in graphs.items():
    for threshold, stages in thresholds.items():
        if "initial" in stages:  # Only count 'initial' graphs
            num_edges = stages["initial"].size()
            rows.append({
                "model": model,
                "threshold": threshold,
                "num_edges_final": num_edges
            })

final_num_edges_df = pd.DataFrame(rows).sort_values(["model", "threshold"]).reset_index(drop=True)
final_num_edges_df

#### Top edges

In [ ]:
es_edge_df = edge_summary.groupby(['edge'], as_index=False).agg(
    model_count=('model', 'nunique'),
    freq_sum=('freq', 'sum')
)

# average frequency across the 5 bootstrap models (adjust divisor if needed)
es_edge_df['freq_avg'] = es_edge_df['freq_sum'] / 5

# Use parentheses around each comparison and compare against freq_avg (not freq_sum)
es_edge_df['strong_consensus'] = (es_edge_df['model_count'] >= 4) & (es_edge_df['freq_avg'] >= 0.4)
es_edge_df['moderate_consensus'] = (es_edge_df['model_count'] >= 3) & (es_edge_df['freq_avg'] >= 0.3)
es_edge_df['weak_consensus'] = (es_edge_df['model_count'] >= 2) & (es_edge_df['freq_avg'] >= 0.2)

es_edge_df.head(3)

In [ ]:
print(f"Total number of edges identified acros models: {len(es_edge_df)}")
print(f"Number of edges with strong consensus: {len(es_edge_df[es_edge_df['strong_consensus']==True])}")
print(f"Number of edges with moderate consensus: {len(es_edge_df[es_edge_df['moderate_consensus']==True])}")
print(f"Number of edges with weak consensus: {len(es_edge_df[es_edge_df['weak_consensus']==True])}")

In [ ]:
es_edge_df[es_edge_df['strong_consensus']==True].sort_values('freq_avg', ascending=False)

Conclusions: Many of the edges with strong consensus can be explained by their temporal links (ex: pre_sleep and acute_sleep), or their biological similarities (ex: heart rate variability and root mean square of successive differences between normal heartbeats). There are however some curious relationships that were found (ex: long_covid and me_cfs). 

#### Model edge frequencies

In [ ]:
edge_summary_clean = edge_summary.copy()
edge_summary_clean['model'].replace({'pc':'PC', 'fci':'FCI', 'lingam':'Linear Mixed', 'notears_linear':'NOTEARS linear', 'notears_nonlinear':'NOTEARS nonlinear'}, inplace=True)

# 1. Violin plot for distribution shape
plt.figure(figsize=(10, 6))
sns.violinplot(data=edge_summary_clean, x="model", y="freq", cut=0)
plt.title("Edge frequency density per algorithm")
plt.xlabel("Algorithm")
plt.ylabel("Edge frequency")
plt.show()

# 2. ECDF plot (very interpretable)
plt.figure(figsize=(8, 6))
for m, df in edge_summary_clean.groupby("model"):
    x = df["freq"].sort_values()
    y = np.arange(1, len(x)+1) / len(x)
    plt.step(x, y, where='post', label=m)

plt.title("ECDF of edge frequencies per algorithm")
plt.xlabel("Edge frequency")
plt.ylabel("Proportion ≤ frequency")
plt.legend()
plt.show()

# 3. Bar plot of number of edges per model
counts = edge_summary_clean.groupby("model")["edge"].nunique().reset_index()
plt.figure(figsize=(10, 6))
plt.bar(counts["model"], counts["edge"])
plt.title(f"Number of unique edges found over {bootstrap} bootstraps")
plt.xlabel("Algorithm")
plt.show()

In [ ]:
es_model_thr = edge_summary.groupby(['model'], as_index=False).agg(
    num_edges=('edge', 'nunique'), freq_avg=('freq','mean'), freq_std=('freq','std'), freq_median=('freq','median'))

es_model_thr

In [ ]:
es_model_thr_2 = edge_summary[edge_summary['freq']>=0.2].groupby(['model'], as_index=False).agg(
    num_edges=('edge', 'nunique'), freq_avg=('freq','mean'), freq_median=('freq','median'))

es_model_thr_35 = edge_summary[edge_summary['freq']>=0.35].groupby(['model'], as_index=False).agg(
    num_edges=('edge', 'nunique'), freq_avg=('freq','mean'), freq_median=('freq','median'))

es_model_thr_5 = edge_summary[edge_summary['freq']>=0.5].groupby(['model'], as_index=False).agg(
    num_edges=('edge', 'nunique'), freq_avg=('freq','mean'), freq_median=('freq','median'))

es_model_thr_8 = edge_summary[edge_summary['freq']>=0.8].groupby(['model'], as_index=False).agg(
    num_edges=('edge', 'nunique'), freq_avg=('freq','mean'), freq_median=('freq','median'))

edges_over_thr = es_model_thr[['model', 'num_edges']].merge(
es_model_thr_2[['model', 'num_edges']].rename(columns={'num_edges':'num_edges_thr_20'}), how='inner', on='model').merge(
    es_model_thr_35[['model', 'num_edges']].rename(columns={'num_edges':'num_edges_thr_35'}), how='inner', on='model').merge(
        es_model_thr_5[['model', 'num_edges']].rename(columns={'num_edges':'num_edges_thr_50'}), how='inner', on='model').merge(
            es_model_thr_8[['model', 'num_edges']].rename(columns={'num_edges':'num_edges_thr_80'}), how='inner', on='model')
edges_over_thr['pct_edges_over_20'] = edges_over_thr['num_edges_thr_20'] / edges_over_thr['num_edges']
edges_over_thr['pct_edges_over_35'] = edges_over_thr['num_edges_thr_35'] / edges_over_thr['num_edges']
edges_over_thr['pct_edges_over_50'] = edges_over_thr['num_edges_thr_50'] / edges_over_thr['num_edges']
edges_over_thr['pct_edges_over_80'] = edges_over_thr['num_edges_thr_80'] / edges_over_thr['num_edges']

edges_over_thr

In [ ]:
edges_thr_02 = edge_summary[edge_summary['freq']>=0.2][['model', 'edge']].groupby(['model'], as_index = False).agg(num_edges = ('edge','nunique'))
edges_thr_02['threshold'] = 0.2
edges_thr_035 = edge_summary[edge_summary['freq']>=0.35][['model', 'edge']].groupby(['model'], as_index = False).agg(num_edges = ('edge','nunique'))
edges_thr_035['threshold'] = 0.35
edges_thr_05 = edge_summary[edge_summary['freq']>=0.5][['model', 'edge']].groupby(['model'], as_index = False).agg(num_edges = ('edge','nunique'))
edges_thr_05['threshold'] = 0.5

edges_per_model = pd.concat([edges_thr_02, edges_thr_035, edges_thr_05], sort=False).sort_values(['model', 'threshold'])
edges_per_model = edges_per_model[['model', 'threshold', 'num_edges']].merge(final_num_edges_df, how='left', on=['model', 'threshold'])
edges_per_model['final_edge_pct'] = edges_per_model['num_edges_final'] / edges_per_model['num_edges']
edges_per_model

### Path summary

In [ ]:
from itertools import islice

def analyze_causal_pathways_bootstrap(df, 
                                     source_vars,
                                     target_vars,
                                     models=None,
                                     max_path_length=None):
    """
    Analyze pathway stability across bootstrap samples for multiple models.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: 'model', 'bootstrap', 'edge'
        where edge is a tuple like ('source', 'target')
    source_vars : list
        List of source/treatment variables (e.g., ['infection_episode'])
    target_vars : list
        List of outcome variables (e.g., ['post_symp_sev_prop', ...])
    models : list, optional
        List of models to analyze. If None, analyzes all models in df
    max_path_length : int, optional
        Maximum path length to consider. If None, no limit
    
    Returns:
    --------
    dict : Nested dictionary with results for each model and source-target pair
    """
    
    if models is None:
        models = df['model'].unique()
    
    results = {}
    
    for model in models:
        print(f"\n{'='*60}")
        print(f"Analyzing: {model}")
        print(f"{'='*60}")
        
        model_df = df[df['model'] == model].copy()
        bootstraps = model_df['bootstrap'].unique()
        n_bootstraps = len(bootstraps)
        
        print(f"Number of bootstraps: {n_bootstraps}")
        
        model_results = {
            'n_bootstraps': n_bootstraps,
            'source_target_pairs': {}
        }
        
        # Analyze each source-target pair
        for source in source_vars:
            for target in target_vars:
                
                print(f"\n--- {source} → {target} ---")
                
                pair_key = f"{source} → {target}"
                pair_results = {
                    'path_existence': 0,  # count of bootstraps with ANY path
                    'direct_edge': 0,     # count of bootstraps with direct edge
                    'all_paths': [],      # store all unique paths found
                    'path_counts': Counter(),  # count occurrences of each path
                    'path_lengths': [],   # distribution of path lengths
                    'mediator_frequency': Counter()  # how often each node mediates
                }
                
                # Analyze each bootstrap
                for bootstrap_id in bootstraps:
                    boot_edges = model_df[model_df['bootstrap'] == bootstrap_id]['edge'].tolist()
                    
                    # Build graph for this bootstrap
                    G_boot = nx.DiGraph()
                    G_boot.add_edges_from(boot_edges)
                    
                    # Check for direct edge
                    if G_boot.has_edge(source, target):
                        pair_results['direct_edge'] += 1
                    
                    # Find all paths (with length limit if specified)
                    try:
                        if max_path_length:
                            paths = list(nx.all_simple_paths(G_boot, source, target, 
                                                            cutoff=max_path_length))
                        else:
                            paths = list(nx.all_simple_paths(G_boot, source, target))
                        
                        if paths:
                            pair_results['path_existence'] += 1
                            
                            for path in paths:
                                # Store path as tuple for counting
                                path_tuple = tuple(path)
                                pair_results['path_counts'][path_tuple] += 1
                                pair_results['path_lengths'].append(len(path))
                                
                                # Count mediators (nodes between source and target)
                                mediators = path[1:-1]
                                for mediator in mediators:
                                    pair_results['mediator_frequency'][mediator] += 1
                    
                    except nx.NetworkXNoPath:
                        continue
                    except Exception as e:
                        print(f"Warning: Error finding paths in bootstrap {bootstrap_id}: {e}")
                        continue
                
                # Calculate percentages
                pair_results['path_existence_pct'] = 100 * pair_results['path_existence'] / n_bootstraps
                pair_results['direct_edge_pct'] = 100 * pair_results['direct_edge'] / n_bootstraps
                
                # Identify most common paths
                if pair_results['path_counts']:
                    total_path_instances = sum(pair_results['path_counts'].values())
                    pair_results['most_common_paths'] = [
                        {
                            'path': ' → '.join(path),
                            'count': count,
                            'pct_of_bootstraps': 100 * count / n_bootstraps,
                            'pct_of_path_instances': 100 * count / total_path_instances
                        }
                        for path, count in pair_results['path_counts'].most_common(10)
                    ]
                else:
                    pair_results['most_common_paths'] = []
                
                # Path length statistics
                if pair_results['path_lengths']:
                    pair_results['path_length_stats'] = {
                        'mean': np.mean(pair_results['path_lengths']),
                        'median': np.median(pair_results['path_lengths']),
                        'min': min(pair_results['path_lengths']),
                        'max': max(pair_results['path_lengths']),
                        'total_paths_found': len(pair_results['path_lengths'])
                    }
                else:
                    pair_results['path_length_stats'] = None
                
                # Top mediators
                if pair_results['mediator_frequency']:
                    pair_results['top_mediators'] = [
                        {
                            'node': node,
                            'appearances': count,
                            'pct_of_bootstraps': 100 * count / n_bootstraps
                        }
                        for node, count in pair_results['mediator_frequency'].most_common(10)
                    ]
                else:
                    pair_results['top_mediators'] = []
                
                # Print summary
                print(f"  Path existence: {pair_results['path_existence_pct']:.1f}% of bootstraps")
                print(f"  Direct edge: {pair_results['direct_edge_pct']:.1f}% of bootstraps")
                if pair_results['path_length_stats']:
                    print(f"  Total unique paths found: {len(pair_results['path_counts'])}")
                    print(f"  Mean path length: {pair_results['path_length_stats']['mean']:.1f}")
                    print(f"  Path length range: {pair_results['path_length_stats']['min']}-{pair_results['path_length_stats']['max']}")
                
                model_results['source_target_pairs'][pair_key] = pair_results
        
        results[model] = model_results
    
    return results


def print_pathway_report(results, source_var, target_var, top_n_paths=5):
    """
    Print a formatted report for a specific source-target pair across all models.
    """
    pair_key = f"{source_var} → {target_var}"
    
    print(f"\n{'='*80}")
    print(f"PATHWAY ANALYSIS REPORT: {pair_key}")
    print(f"{'='*80}\n")
    
    for model, model_data in results.items():
        if pair_key not in model_data['source_target_pairs']:
            continue
            
        pair_data = model_data['source_target_pairs'][pair_key]
        n_bootstraps = model_data['n_bootstraps']
        
        print(f"\n{'-'*80}")
        print(f"Model: {model}")
        print(f"{'-'*80}")
        
        print(f"\n1. PATH EXISTENCE:")
        print(f"   - Any path exists: {pair_data['path_existence_pct']:.1f}% ({pair_data['path_existence']}/{n_bootstraps} bootstraps)")
        print(f"   - Direct edge exists: {pair_data['direct_edge_pct']:.1f}% ({pair_data['direct_edge']}/{n_bootstraps} bootstraps)")
        
        if pair_data['path_existence'] > 0:
            indirect_pct = pair_data['path_existence_pct'] - pair_data['direct_edge_pct']
            print(f"   - Indirect paths only: {indirect_pct:.1f}%")
        
        if pair_data['path_length_stats']:
            stats = pair_data['path_length_stats']
            print(f"\n2. PATH CHARACTERISTICS:")
            print(f"   - Total unique paths: {len(pair_data['path_counts'])}")
            print(f"   - Total path instances: {stats['total_paths_found']}")
            print(f"   - Mean length: {stats['mean']:.1f} nodes")
            print(f"   - Median length: {stats['median']:.1f} nodes")
            print(f"   - Range: {stats['min']}-{stats['max']} nodes")
        
        if pair_data['most_common_paths']:
            print(f"\n3. TOP {min(top_n_paths, len(pair_data['most_common_paths']))} MOST COMMON PATHWAYS:")
            for i, path_info in enumerate(pair_data['most_common_paths'][:top_n_paths], 1):
                print(f"   {i}. {path_info['path']}")
                print(f"      → Appears in {path_info['pct_of_bootstraps']:.1f}% of bootstraps ({path_info['count']} times)")
        
        if pair_data['top_mediators']:
            print(f"\n4. TOP 10 MEDIATING NODES:")
            for i, med_info in enumerate(pair_data['top_mediators'][:10], 1):
                print(f"   {i}. {med_info['node']}")
                print(f"      → Appears in {med_info['pct_of_bootstraps']:.1f}% of bootstraps ({med_info['appearances']} times)")


def compare_models_pathways(results, source_var, target_var):
    """
    Compare pathway characteristics across different models.
    """
    pair_key = f"{source_var} → {target_var}"
    
    print(f"\n{'='*80}")
    print(f"CROSS-MODEL COMPARISON: {pair_key}")
    print(f"{'='*80}\n")
    
    comparison_data = []
    
    for model, model_data in results.items():
        if pair_key not in model_data['source_target_pairs']:
            continue
        
        pair_data = model_data['source_target_pairs'][pair_key]
        
        row = {
            'Model': model,
            'Path Exists (%)': pair_data['path_existence_pct'],
            'Direct Edge (%)': pair_data['direct_edge_pct'],
            'Unique Paths': len(pair_data['path_counts']) if pair_data['path_counts'] else 0,
            'Mean Path Length': pair_data['path_length_stats']['mean'] if pair_data['path_length_stats'] else None
        }
        comparison_data.append(row)
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.to_string(index=False))
    
    # Find consensus pathways (appear in multiple models)
    print(f"\n{'='*80}")
    print("CONSENSUS PATHWAYS (across models)")
    print(f"{'='*80}\n")
    
    all_paths = Counter()
    for model, model_data in results.items():
        if pair_key in model_data['source_target_pairs']:
            pair_data = model_data['source_target_pairs'][pair_key]
            for path in pair_data['path_counts'].keys():
                all_paths[path] += 1
    
    n_models = len([m for m in results.keys() if pair_key in results[m]['source_target_pairs']])
    
    print("Pathways found in multiple models:")
    consensus_paths = [(path, count) for path, count in all_paths.items() if count > 1]
    consensus_paths.sort(key=lambda x: x[1], reverse=True)
    
    if consensus_paths:
        for path, n_models_with_path in consensus_paths[:20]:  # top 20
            print(f"  {' → '.join(path)}")
            print(f"    → Found in {n_models_with_path}/{n_models} models")
    else:
        print("  No pathways found in multiple models")
    
    return comparison_df


# =============================================================================
# MAIN ANALYSIS EXECUTION
# =============================================================================

# Load your data
# df should have columns: 'model', 'bootstrap', 'edge'
# where edge is stored as tuples like ('source', 'target')

# RQ1: infection_episode → symptoms
print("\n" + "="*80)
print("RESEARCH QUESTION 1: How do cumulative infections influence symptoms?")
print("="*80)

rq1_outcomes = ['post_num_symp_prop', 'post_symp_sev_prop', 'post_symp_freq_prop']
rq1_results = analyze_causal_pathways_bootstrap(
    df=df,
    source_vars=['infection_episode'],
    target_vars=rq1_outcomes,
    models=None,  # analyze all models
    max_path_length=None  # no limit initially
)

# Print detailed reports for each outcome
for outcome in rq1_outcomes:
    print_pathway_report(rq1_results, 'infection_episode', outcome, top_n_paths=10)
    compare_models_pathways(rq1_results, 'infection_episode', outcome)

# RQ2: infection_episode → other outcomes
print("\n" + "="*80)
print("RESEARCH QUESTION 2: How do cumulative infections influence other outcomes?")
print("="*80)

# Define other outcomes (all acute_ and post_ variables except symptoms)
other_outcomes = []
for col in final_df_std.columns:
    if (col.startswith("acute") or col.startswith("post")) and (col not in rq1_outcomes):
        other_outcomes.append(col)

print(f"Analyzing {len(other_outcomes)} other outcome variables...")

rq2_results = analyze_causal_pathways_bootstrap(
    df=df,
    source_vars=['infection_episode'],
    target_vars=other_outcomes,
    models=None,
    max_path_length=None
)

# Print summary for top outcomes (ones with paths)
print("\n" + "="*80)
print("RQ2 SUMMARY: Outcomes with stable pathways from infection_episode")
print("="*80)

for model in rq2_results.keys():
    print(f"\nModel: {model}")
    outcomes_with_paths = []
    
    for pair_key, pair_data in rq2_results[model]['source_target_pairs'].items():
        if pair_data['path_existence_pct'] > 50:  # paths in >50% of bootstraps
            target = pair_key.split(' → ')[1]
            outcomes_with_paths.append({
                'Outcome': target,
                'Path Exists (%)': pair_data['path_existence_pct'],
                'Direct Edge (%)': pair_data['direct_edge_pct'],
                'Unique Paths': len(pair_data['path_counts'])
            })
    
    if outcomes_with_paths:
        summary_df = pd.DataFrame(outcomes_with_paths)
        summary_df = summary_df.sort_values('Path Exists (%)', ascending=False)
        print(summary_df.to_string(index=False))
    else:
        print("  No outcomes with stable pathways (>50% of bootstraps)")

# RQ3: other factors → symptoms
print("\n" + "="*80)
print("RESEARCH QUESTION 3: How do other factors influence symptoms?")
print("="*80)

# Define other treatments
other_treatments = []
for col in final_df_std.columns:
    if ((col.count("symp")==0) and (col.startswith("acute") or col.startswith("pre"))) or (col in [
        "age", "is_female", "long_covid", "me_cfs", "dysautonomia", "fibromyalgia", "period_at_covid_start"]):
        other_treatments.append(col)

print(f"Analyzing {len(other_treatments)} treatment variables...")

rq3_results = analyze_causal_pathways_bootstrap(
    df=df,
    source_vars=other_treatments,
    target_vars=rq1_outcomes,  # same symptom outcomes as RQ1
    models=None,
    max_path_length=None
)

# Print summary for top treatments (ones with stable paths)
print("\n" + "="*80)
print("RQ3 SUMMARY: Factors with stable pathways to symptoms")
print("="*80)

for model in rq3_results.keys():
    print(f"\nModel: {model}")
    
    for outcome in rq1_outcomes:
        print(f"\n  Target: {outcome}")
        treatments_with_paths = []
        
        for pair_key, pair_data in rq3_results[model]['source_target_pairs'].items():
            if outcome in pair_key and pair_data['path_existence_pct'] > 50:
                source = pair_key.split(' → ')[0]
                treatments_with_paths.append({
                    'Source': source,
                    'Path Exists (%)': pair_data['path_existence_pct'],
                    'Direct Edge (%)': pair_data['direct_edge_pct'],
                    'Unique Paths': len(pair_data['path_counts'])
                })
        
        if treatments_with_paths:
            summary_df = pd.DataFrame(treatments_with_paths)
            summary_df = summary_df.sort_values('Path Exists (%)', ascending=False)
            print(summary_df.to_string(index=False))
        else:
            print("    No factors with stable pathways (>50% of bootstraps)")

# Save detailed results
import pickle
with open('pathway_analysis_results.pkl', 'wb') as f:
    pickle.dump({
        'rq1': rq1_results,
        'rq2': rq2_results,
        'rq3': rq3_results
    }, f)

print("\n" + "="*80)
print("Analysis complete! Results saved to 'pathway_analysis_results.pkl'")
print("="*80)

### Fit summary

In [ ]:
ft_model_stg_thr_df = fit_summary.merge(final_num_edges_df, how='left', on=['model', 'threshold']).groupby(['model', 'threshold', 'stage', 'num_edges_final'], as_index=False).agg(
        r2=('r2','mean'), crps=('crps','mean'), f1=('f1','mean'), kl=('kl','mean'), n_nodes_passed=('passed_metric_threshold','sum'), n_nodes=('node','nunique'))
ft_model_stg_thr_df['pct_nodes_passed'] = ft_model_stg_thr_df['n_nodes_passed']/ft_model_stg_thr_df['n_nodes']

ft_model_stg_thr_df.head(3)

In [ ]:
# Pivot and calculate deltas
ft_model_thr_df = ft_model_stg_thr_df.pivot(index=['model', 'threshold', 'num_edges_final'], columns='stage', values=[
    'r2', 'crps', 'f1', 'kl', 'n_nodes_passed', 'n_nodes', 'pct_nodes_passed'])

ft_model_thr_df['delta_r2'] = ft_model_thr_df['r2']['post node refinement'] - ft_model_thr_df['r2']['initial']
ft_model_thr_df['delta_crps'] = ft_model_thr_df['crps']['post node refinement'] - ft_model_thr_df['crps']['initial']
ft_model_thr_df['delta_f1'] = ft_model_thr_df['f1']['post node refinement'] - ft_model_thr_df['f1']['initial']
ft_model_thr_df['delta_kl'] = ft_model_thr_df['kl']['post node refinement'] - ft_model_thr_df['kl']['initial']

ft_model_thr_df.columns = ['_'.join([str(c) for c in col if c]) for col in ft_model_thr_df.columns]
ft_model_thr_df = ft_model_thr_df.reset_index()

ft_model_thr_df

In [ ]:
node_log_summary['metric'].unique()

In [ ]:
ft_model_thr_df[['model', 'threshold', 'kl_initial', 'kl_post node refinement', 'delta_kl']]

In [ ]:
ft_model_thr_df[['model', 'threshold', 'num_edges_final', 'crps_post node refinement']]

In [ ]:
# Pivot and calculate deltas
ft_model_thr_df = ft_model_stg_thr_df.pivot(index=['model', 'threshold', 'num_edges_final'], columns='stage', values=[
    'r2', 'crps', 'f1', 'kl', 'n_nodes_passed', 'n_nodes', 'pct_nodes_passed'])

ft_model_thr_df['delta_r2'] = ft_model_thr_df['r2']['post node refinement'] - ft_model_thr_df['r2']['initial']
ft_model_thr_df['delta_crps'] = ft_model_thr_df['crps']['post node refinement'] - ft_model_thr_df['crps']['initial']
ft_model_thr_df['delta_f1'] = ft_model_thr_df['f1']['post node refinement'] - ft_model_thr_df['f1']['initial']
ft_model_thr_df['delta_kl'] = ft_model_thr_df['kl']['post node refinement'] - ft_model_thr_df['kl']['initial']

ft_model_thr_df.columns = ['_'.join([str(c) for c in col if c]) for col in ft_model_thr_df.columns]
ft_model_thr_df = ft_model_thr_df.reset_index()

ft_model_thr_df

### Causal Structure

In [ ]:
# Graphs to assess further

fci_02_g = graphs['fci'][0.2]['refined']
fci_035_g = graphs['fci'][0.35]['refined']
fci_05_g = graphs['fci'][0.5]['refined']
notears_nonlinear_02_g = graphs['notears_nonlinear'][0.2]['refined']
notears_nonlinear_035_g = graphs['notears_nonlinear'][0.35]['refined']
notears_nonlinear_05_g = graphs['notears_nonlinear'][0.5]['refined']

models_to_assess = {'fci_02_g':fci_02_g, 'fci_035_g':fci_035_g, 'fci_05_g':fci_05_g, 'notears_nonlinear_02_g':notears_nonlinear_02_g, 
'notears_nonlinear_035_g':notears_nonlinear_035_g, 'notears_nonlinear_05_g':notears_nonlinear_05_g}

In [ ]:
# causal_struct_df = []

# for model_name, model in models_to_assess.items():

#     # Create SCM with original mechanisms
#     scm = gcm.InvertibleStructuralCausalModel(model)

#     # Fit the causal mechanisms to the data
#     gcm.fit(scm, final_df_std)

#     # Run evaluation
#     eval_result_struct = gcm.evaluate_causal_model(
#         scm, final_df_std,
#         evaluate_causal_mechanisms=False,  # calculates normalized continuous ranked probability score (all nodes), MSE, normalized MSE, and R2 (continuous), and F1 (categorical)
#         compare_mechanism_baselines=False, # compares the causal mechanisms with baseline models to see if there are model choices that perform significantly better
#         evaluate_invertibility_assumptions=False, # tests statistical independence between the reconstructed noise and the used input samples
#         evaluate_overall_kl_divergence=False, # tests KL divergence between the generated and the observed data
#         evaluate_causal_structure=True # evaluates to find substantial evidence to refute the causal graph based on the provided data
#     )
#     struct = getattr(eval_result_struct, "graph_falsification", {})

#     # Put together dataset
#     causal_struct_df.append({'model':model_name, 'causal_structure':struct})

In [ ]:
# causal_struct_df

### Choosing best model

In [ ]:
x_nt = [0,0.2,0.35,0.5,0.8]
y_nt = [1.0, 0.641509, 0.550943, 0.498113, 0.352830]

plt.plot(x_nt,y_nt, marker='o')
plt.xlabel("Thresholds")        # Label for the X-axis
plt.ylabel("Percent of edges retained")        # Label for the Y-axis
plt.title("NOTEARS Nonlinear Edge Stability")  # Chart title
plt.show()

### Node log / Benchmark summary

In [ ]:
best_models = {'notears_nonlinear':[0.35]}

In [ ]:
best_model = 'notears_nonlinear'
best_threshold = 0.35
best_key_str = f"{best_model}_b{bootstrap}_{best_threshold}".replace(".", "_")

In [ ]:
benchmark_summary_best = benchmark_summary[
            (benchmark_summary["model"] == best_model) & (benchmark_summary["threshold"] == threshold)
        ][["model", "threshold", "node", 'scm_r2', 'benchmark_r2', 'benchmark_top_parents']].dropna()
benchmark_summary_best

In [ ]:
benchmark_summary_best['scm_r2'].mean()

In [ ]:
benchmark_summary_best['benchmark_r2'].mean()

In [ ]:
nodes_refined_best = node_log_summary[(node_log_summary['model']==best_model)&(node_log_summary['threshold']==best_threshold)]['node'].unique().tolist()

benchmark_summary[(benchmark_summary['model']==best_model)&(benchmark_summary['threshold']==best_threshold)&
            (benchmark_summary['node'].isin(nodes_refined_best))][['node', 'benchmark_r2', 'benchmark_top_parents']]

## Analysis on the best graph

### Publishing interactive view

In [ ]:
# Load best graph
G_best = graphs[best_model][best_threshold]['refined']

#  Load SCM of best graph
scm_best = scms[best_model][best_threshold]['refined_scm']

In [ ]:
def extract_from_graph(G):
    # Ensure deterministic node order
    node_labels = list(G.nodes())
    index = {node: i for i, node in enumerate(node_labels)}
    n = len(node_labels)
    
    adj_mat = np.zeros((n, n))
    for u, v in G.edges():
        adj_mat[index[u], index[v]] = 1   # or any weight you prefer

    return adj_mat, node_labels

In [ ]:
from pyvis.network import Network

def plot_interactive_graph_phases(adj_mat, node_labels, threshold, img_file, known_bk_edges):
    
    # --- group by phase ---
    def assign_group(name):
        if "pre" in name:
            return "pre"
        elif "acute" in name:
            return "acute"
        elif "post" in name:
            return "post"
        else:
            return "other"
    
    node_groups = [assign_group(name) for name in node_labels]

    GROUP_COLORING = {
        "pre": "skyblue",
        "acute": "lightgreen",
        "post": "salmon",
        "other": "gray"
    }

    net = Network(height="1000px", width="100%", directed=True, notebook=True)
    net.set_options("""
        {
        "physics": {
            "enabled": false
        },
        "edges": {
            "arrows": {
            "to": {"enabled": true, "scaleFactor": 1.0}
            },
            "smooth": false
        },
        "interaction": {
            "dragNodes": true
        }
        }
        """)

    n = len(node_labels)
    unique_groups = sorted(set(node_groups))

    # Layout
    groups_per_row = 3
    group_spacing_x = 900
    group_spacing_y = 900
    max_nodes_per_row = 8
    node_spacing_x = 150
    node_spacing_y = 120

    # Place group blocks
    group_centers = {}
    for idx, grp in enumerate(unique_groups):
        row_idx = idx // groups_per_row
        col_idx = idx % groups_per_row
        group_centers[grp] = (col_idx * group_spacing_x, row_idx * group_spacing_y)

    from collections import defaultdict
    group_indices = defaultdict(int)

    # Add nodes
    for i in range(n):
        grp = node_groups[i]
        center_x, center_y = group_centers[grp]
        local_idx = group_indices[grp]
        group_indices[grp] += 1
        col = local_idx % max_nodes_per_row
        row = local_idx // max_nodes_per_row
        pos_x = center_x + col * node_spacing_x
        pos_y = center_y + row * node_spacing_y

        net.add_node(
            i,
            label=node_labels[i],
            title=node_labels[i],
            x=pos_x,
            y=pos_y,
            color=GROUP_COLORING[grp],
            shape="ellipse",
            font={"align": "center", "size": 16, "color": "black"}
        )

    # Add edges
    number_of_BK_edges = 0
    number_of_non_BK_edges = 0
    
    for i in range(n):
        for j in range(n):
            if adj_mat[i][j] > threshold or adj_mat[i][j] < -threshold:
                if [i, j] in known_bk_edges:
                    edge_color = "black"
                    number_of_BK_edges += 1
                else:
                    edge_color = "blue"
                    number_of_non_BK_edges += 1

                net.add_edge(i, j, arrows="to", color=edge_color, width=2)

    print(f"tr={threshold}, bk_edges={number_of_BK_edges}, non_bk_edges={number_of_non_BK_edges}")

    net.show(f"{img_file}_{threshold}.html")

In [ ]:
adj_mat, node_labels = extract_from_graph(G_best)

required_edges_str = [
    ("age", "period_at_covid_start"),
    ("is_female", "period_at_covid_start"),
    ("is_female", "me_cfs")
]
def encode_required_edges(required_edges_str, node_labels):
    label_to_index = {lbl: i for i, lbl in enumerate(node_labels)}
    encoded = []

    for src, tgt in required_edges_str:
        if src not in label_to_index:
            print(f"I cannot confirm this: missing node '{src}' in graph.")
            continue
        if tgt not in label_to_index:
            print(f"I cannot confirm this: missing node '{tgt}' in graph.")
            continue

        encoded.append([label_to_index[src], label_to_index[tgt]])

    return encoded

known_bk_edges = encode_required_edges(required_edges_str, node_labels)
known_bk_edges

plot_interactive_graph_phases(
    adj_mat=adj_mat,
    node_labels=node_labels,
    threshold=0.35,            # whatever threshold you use
    img_file="causal_graph",
    known_bk_edges=known_bk_edges
)

In [ ]:
os.path.abspath("causal_graph_0.35.html")

In [ ]:
def graph_edge_analysis(G):
    
    # 1. Edge Stability Analysis
    edge_weights = nx.get_edge_attributes(G, 'weight')

    edges = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges()]
    highly_stable = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges() if edge_weights.get((u,v),0) >= .85]
    moderately_stable = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges() if edge_weights.get((u,v),0) >= .6 and edge_weights.get((u,v),0) < .85]
    unstable = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges() if edge_weights.get((u,v),0) >= .35 and edge_weights.get((u,v),0) < .6]
    print(f"\nHighly stable edges (≥85%): {len(highly_stable)}")
    print(f"Moderately stable edges (60-84%): {len(moderately_stable)}")
    print(f"Unstable edges (<60%): {len(unstable)}")

    # 2. Critical Bottlenecks
    # Find nodes that are "hubs" in the stable network
    # Count stable edges in/out of each node

    outgoing_stable = Counter(u for u,v,w in highly_stable)
    incoming_stable = Counter(v for u,v,w in highly_stable)

    outgoing = Counter(u for u,v,w in edges)
    incoming = Counter(v for u,v,w in edges)

    print("\nNodes with most edge frequencies over 85%:")
    print("Top 10 outgoing:")
    for node, count in outgoing_stable.most_common(10):
        print(f"  {node}: {count} stable outgoing edges")

    print("\nTop 10 incoming:")
    for node, count in incoming_stable.most_common(10):
        print(f"  {node}: {count} stable incoming edges")

    # 3. Most Stable Edges
    print("\nEdges with most stable frequencies:")
    print("Top 75 edges:")
    for u,v,w in sorted(highly_stable, key=lambda x: (-x[2]))[:75]:
        print(u,v,w)

In [ ]:
graph_edge_analysis(G_best)

### Intervention analysis

In [ ]:
name_map = {'infection_episode':'Infection Episode', 'age':'Age', 'is_female':'Is Female', 'long_covid':'Long COVID', 'me_cfs':'ME/CFS', 'fibromyalgia':'Fibromyalgia',
            'dysautonomia':'Dysautonomia', 'period_at_covid_start':'Period at Infection Start', 'pre_emotionally_stressful':'Pre-Infection Emotional Stress',
            'pre_hr_variability': 'Pre-Infection HRV', 'pre_mentally_demanding':'Pre-Infection Mental Exertion', 'pre_physically_active':'Pre-Infection Physical Exertion',
            'pre_resting_hr':'Pre-Infection Resting Heart Rate', 'pre_sleep':'Pre-Infection Sleep Quality', 'acute_emotionally_stressful':'Acute Infection Emotional Stress',
            'acute_hr_variability':'Acute Infection HRV', 'acute_mentally_demanding':'Acute Infection Mental Exertion', 'acute_physically_active':'Acute Infection Physical Exertion',
            'acute_resting_hr':'Acute Infection Resting Heart Rate', 'acute_sleep':'Acute Infection Sleep Quality', 'post_emotionally_stressful':'Post-Infection Emotional Stress',
            'post_hr_variability':'Post-Infection HRV', 'post_mentally_demanding':'Post-Infection Mental Exertion', 'post_physically_active':'Post-Infection Physical Exertion',
            'post_resting_hr':'Post-Infection Resting Heart Rate', 'post_sleep':'Post-Infection Sleep Quality', 'pre_funcap_score':'Pre-Infection Functional Capacity', 
            'post_funcap_score':'Post-Infection Functional Capacity', 'pre_crash':'Pre-Infection Crash', 'pre_noncovid_infection':'Pre-Infection Non-COVID Infection', 
            'acute_crash':'Acute Infection Crash', 'acute_noncovid_infection':'Acute Infection Non-COVID Infection', 'post_crash':'Post-Infection Crash', 
            'post_noncovid_infection':'Post-Infection Non-COVID Infection', 'pre_num_symp_prop':'Pre-Infection Number of Symptoms', 'pre_symp_sev_prop':'Pre-Infection Symptom Severity', 
            'pre_symp_freq_prop':'Pre-Infection Symptom Frequency', 'acute_num_symp_prop':'Acute Infection Number of Symptoms', 'acute_symp_sev_prop':'Acute Infection Symptom Severity', 
            'acute_symp_freq_prop':'Acute Infection Symptom Frequency', 'post_num_symp_prop':'Post-Infection Number of Symptoms', 'post_symp_sev_prop':'Post-Infection Symptom Severity', 
            'post_symp_freq_prop':'Post-Infection Symptom Frequency'}

In [ ]:
def build_forward_inverse_from_registry(r):
    t = r["type"]
    method = r["log_method"]
    eps = r.get("eps", 0)
    shift = r.get("shift", 0)
    max_val = r.get("max_value", None)
    scaler = r.get("scaler", None)

    # --- PROPORTIONS ---
    if t == "proportion":
        if method == "logit":
            def fwd(x):
                z = np.log(x.clip(eps,1-eps)/(1-x.clip(eps,1-eps)))
                return scaler.transform(z.reshape(-1,1)).ravel()
            def inv(z):
                unscaled = scaler.inverse_transform(z.reshape(-1,1)).ravel()
                return 1/(1+np.exp(-unscaled))
            return fwd, inv

        else:  # no logit
            def fwd(x):
                return scaler.transform(x.reshape(-1,1)).ravel()
            def inv(z):
                return scaler.inverse_transform(z.reshape(-1,1)).ravel()
            return fwd, inv

    # --- CONTINUOUS / DISCRETE ---
    if method == "log":
        def fwd(x):
            return scaler.transform(np.log(x+1e-6).reshape(-1,1)).ravel()
        def inv(z):
            unscaled = scaler.inverse_transform(z.reshape(-1,1)).ravel()
            return np.exp(unscaled)-1e-6
        return fwd, inv

    if method == "log1p":
        def fwd(x):
            return scaler.transform(np.log1p(x - shift).reshape(-1,1)).ravel()
        def inv(z):
            unscaled = scaler.inverse_transform(z.reshape(-1,1)).ravel()
            return np.expm1(unscaled)+shift
        return fwd, inv

    if method == "logcomp":
        def fwd(x):
            return scaler.transform(np.log1p(max_val - x).reshape(-1,1)).ravel()
        def inv(z):
            unscaled = scaler.inverse_transform(z.reshape(-1,1)).ravel()
            return max_val + 1 - np.exp(unscaled)
        return fwd, inv
    
    # --- BINARY ---
    if r["scaler"] is None:
        def fwd(x): 
            return x
        def inv(z): 
            return z
        return fwd, inv

    # no log transform
    def fwd(x):
        return scaler.transform(x.reshape(-1,1)).ravel()
    def inv(z):
        return scaler.inverse_transform(z.reshape(-1,1)).ravel()
    return fwd, inv

def inverse_outcome_transform(col, x, registry):
    _, inv = build_forward_inverse_from_registry(registry[col])
    return inv(np.array(x, dtype=float))

def get_intervention_values_raw(data, treatment, n_points=6):
    """
    Dynamically infer a clean, interpretable intervention grid for a treatment variable.
    - Keeps binary and small discrete variables as-is.
    - For averaged discrete variables (e.g., 0–3 or 0–10 scales), creates rounded discrete steps.
    - For continuous variables, creates evenly spaced values between 5th and 95th percentile.
    """
    series = data[treatment].dropna()
    min_val, max_val = series.min(), series.max()
    unique_vals = series.unique()
    n_unique = series.nunique()

    # --- Case 1: Binary variable
    if n_unique <= 2:
        return sorted(series.unique().astype(float))

    # --- Case 2: True small discrete (integer) scale (e.g., 0–5)
    elif pd.api.types.is_integer_dtype(series) and n_unique < 15:
        return list(sorted(series.unique()))

    # --- Case 3: Averaged discrete variable (e.g., mean of 0–3 or 0–10 ratings)
    elif (max_val - min_val) <= 10 and (max_val - min_val) > 1 and n_unique > 2:
        # Detect the "natural" discrete scale
        step = np.round((max_val - min_val) / 5, 1)  # coarse step
        vals = np.arange(np.floor(min_val), np.ceil(max_val) + step, step)
        vals = np.round(vals).astype(int)
        vals = np.unique(vals[(vals >= np.floor(min_val)) & (vals <= np.ceil(max_val))])

        return vals.tolist()

    # --- Case 4: Continuous variable (e.g., age, HRV)
    else:
        q_low, q_high = series.quantile([0.05, 0.95])
        vals = np.linspace(q_low, q_high, n_points)
        return np.round(vals, 2).tolist()
    
def get_intervention_values_std(data_raw, treatment, registry, n_points=6):
    """
    Compute intervention grid from RAW units, then transform to standardized
    using the saved forward transform.
    """
    raw_vals = get_intervention_values_raw(data_raw, treatment, n_points=n_points)
    print(raw_vals)

    fwd, _ = build_forward_inverse_from_registry(registry[treatment])
 
    return fwd(np.array(raw_vals, dtype=float)).tolist()

In [ ]:
def run_mc_intervention(scm, outcome, treatment, val, data, n_reps=100):
    "Runs interventions with monte_carlo_intervention function"

    return val, monte_carlo_intervention(
        scm,
        {treatment: lambda pre, v=val: v}, # do-operator utilizing range of values v
        outcome=outcome,
        data=data,
        n_reps=n_reps)

def monte_carlo_intervention(scm, interventions, outcome, data, n_reps=100, 
                 random_seed=42):
    """
    Compute intervention effect on an outcome using GCM.
    
    Parameters
    ----------
    scm : gcm.StructuralCausalModel
        Fitted SCM object.
    interventions : dict
        Node name → callable intervention function (pre_values → post_values).
    outcome : str
        Name of the outcome variable to summarize.
    n_reps : int
        Number of repetitions for Monte Carlo.
    random_seed : int
        Seed for reproducibility.
    
    Returns
    -------
    dict
        {"mean": float,
            "std": float,          # standard deviation or analytic SE
            "95ci": (float, float),# 95% CI
            "raw_means": np.array  # all MC replicates if quick_mode=False}
    """
    gcm.util.general.set_random_seed(42)  # Set once

    raw_means = []
    for i in range(n_reps):
        samples = gcm.interventional_samples(
            scm,
            interventions=interventions,
            observed_data=data
        )
        raw_means.append(samples[outcome].mean())
    
    raw_means = np.array(raw_means)
    mean_val = raw_means.mean()
    std_val = raw_means.std(ddof=1)
    ci_lower, ci_upper = np.percentile(raw_means, [2.5, 97.5])
    
    return {
        "mean": mean_val,
        "std": std_val,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "raw_means": raw_means
        }

In [ ]:
def run_effect_and_interventions(G, scm, raw_data, std_data, registry, treatments, outcomes, rq=1, n_reps=100):

    all_results = []
    valid_pairs = []

    # ---- First pass: identify valid treatment–outcome pairs ----
    for treatment in treatments:
        for outcome in outcomes:

            # Naming for plots
            treatment_name = name_map[treatment]
            outcome_name = name_map[outcome]

            # Check for all paths from treatment to outcome
            paths = list(nx.all_simple_paths(G, source=treatment, target=outcome))
            print(f"Paths from {treatment} to {outcome}:", paths)
            if not paths:
                continue # skip the analysis for combinations that aren't causally linked

            # Add treatment and outcome to variable to plot
            valid_pairs.append((treatment, outcome))
            
    # ---- Skip if no valid causal pairs ----
    if not valid_pairs:
        print("No causal effects detected. Nothing to plot.")
        return

    # ---- Create figure only for valid pairs ----
    n_rows = len(valid_pairs)
    fig, axes = plt.subplots(n_rows, 2, figsize=(16, max(3, 3 * n_rows)))
    axes = np.atleast_2d(axes)

    # ---- Second pass: plot only valid pairs ----
    for idx, (treatment, outcome) in enumerate(valid_pairs):

        # ---- GCM INTERVENTION (distributional effects for all values of treatment) ----                    
        print(f"\nPlotting {treatment} → {outcome}")

        # Naming for plots
        treatment_name = name_map[treatment]
        outcome_name = name_map[outcome]

        # Run interventions 
        intervention_vals = get_intervention_values_std(raw_data, treatment, registry) # Use the raw data intervals to produce standardized intervals
        print(intervention_vals)
        res_list = Parallel(n_jobs=6)(
            delayed(run_mc_intervention)(scm, outcome, treatment, val, std_data, n_reps)
            for val in intervention_vals
        )
        results_dict = {val: res for val, res in res_list}
        print(results_dict)

        mean_outcomes = {val: res["mean"] for val, res in results_dict.items()}
        ci_lower_outcomes = {val: res["ci_lower"] for val, res in results_dict.items()}
        ci_upper_outcomes = {val: res["ci_upper"] for val, res in results_dict.items()}

        print(mean_outcomes)
        mean_raw = {float(inverse_outcome_transform(treatment, val, registry)): 
                    float(inverse_outcome_transform(outcome, mean_outcome, registry)) for val, mean_outcome in mean_outcomes.items()}
        ci_low_raw = {float(inverse_outcome_transform(treatment, val, registry)): 
                        float(inverse_outcome_transform(outcome, ci_lower_outcome, registry)) for val, ci_lower_outcome in ci_lower_outcomes.items()}
        ci_high_raw = {float(inverse_outcome_transform(treatment, val, registry)): 
                        float(inverse_outcome_transform(outcome, ci_upper_outcome, registry)) for val, ci_upper_outcome in ci_upper_outcomes.items()}

        # ---- Mean response plot ----
        ax_mean = axes[idx, 1]
        vals = list(mean_raw.keys())
        means = list(mean_raw.values())
        ci_lower = [ci_low_raw[v] for v in vals]
        ci_upper = [ci_high_raw[v] for v in vals]

        ax_mean.plot(vals, means, marker="o", label="Mean outcome")
        ax_mean.fill_between(vals, ci_lower, ci_upper, color="lightblue", alpha=0.3, label="95% CI")
        ax_mean.set_title(f"Mean {outcome_name} vs {treatment_name}")
        ax_mean.set_xlabel(f"{treatment_name} (do-value)")
        ax_mean.set_ylabel(f"Mean {outcome_name}")
        ax_mean.grid(True)
        ax_mean.legend()

        # ---- Distribution plot ----
        ax_hist = axes[idx, 0]
        for val, res in results_dict.items():
            label_val = int(inverse_outcome_transform(treatment, val, registry)) if float(inverse_outcome_transform(treatment, val, registry)).is_integer() \
                else round(float(inverse_outcome_transform(treatment, val, registry)), 2)
            sns.kdeplot(inverse_outcome_transform(outcome, res["raw_means"], registry), fill=True, alpha=0.3, label=f"do={label_val}", ax=ax_hist)
            ax_hist.axvline(inverse_outcome_transform(outcome, res["mean"], registry), linestyle="--", color="k", alpha=0.6)
        ax_hist.set_title(f"{outcome_name}: Distribution of Monte Carlo mean estimates")
        ax_hist.set_xlabel(f"Mean {outcome_name}")
        ax_hist.set_ylabel("Density")
        ax_hist.legend(title=f"{treatment_name}")

        # ---- Streamlined Reporting ----
        effect_df = pd.DataFrame({
            "do_val": list(results_dict.keys()),
            "mean": [res["mean"] for res in results_dict.values()]
        })

        # Compute key metrics
        min_val_std, max_val_std = effect_df["do_val"].iloc[0], effect_df["do_val"].iloc[-1]
        min_mean_std, max_mean_std = effect_df["mean"].iloc[0], effect_df["mean"].iloc[-1]
        min_val, max_val = float(inverse_outcome_transform(treatment, min_val_std, registry)), float(inverse_outcome_transform(treatment, max_val_std, registry))
        min_mean, max_mean = float(inverse_outcome_transform(outcome, min_mean_std, registry)), float(inverse_outcome_transform(outcome, max_mean_std, registry))

        # Dose-response slope
        slope = (max_mean - min_mean) / (max_val - min_val)

        # Total changes
        abs_change_std = max_mean_std - min_mean_std
        abs_change = max_mean - min_mean
        rel_change = ((max_mean / min_mean) - 1)
        pct_points = abs_change * 100

        # Print summary
        print(f"\n{'='*60}")
        print(f"{treatment.upper()} → {outcome.upper()}")
        print(f"{'='*60}")
        print(f"Intervention range: {min_val:.1f} to {max_val:.1f}")
        print(f"\nDose-Response Effect:")
        print(f"  • Per unit increase: {slope:.4f}")
        print(f"  • Total change ({min_val:.0f}→{max_val:.0f}): {abs_change:.4f}")
        print(f"  • Relative change: {rel_change:.2f}")
        print(f"{'='*60}\n")

        # Update the same row in all_results
        all_results.append({
            "treatment": treatment_name,
            "outcome": outcome_name,
            "min_val": min_val,
            "max_val": max_val, 
            "min_mean_std": min_mean_std,
            "max_mean_std": max_mean_std,
            "min_mean": min_mean,
            "max_mean": max_mean,
            "min_ci_lower": ci_lower[0],      # CI for the minimum intervention
            "min_ci_upper": ci_upper[0],
            "max_ci_lower": ci_lower[-1],     # CI for the maximum intervention
            "max_ci_upper": ci_upper[-1],
            "slope": slope,
            "abs_change_std": abs_change_std,
            "abs_change": abs_change,
            "rel_change": rel_change,
            "pct_points": pct_points
            })

    plt.tight_layout()
    plt.show()

    pd.DataFrame(all_results).to_csv(f"summaries/intervention_summary_{best_model}_rq{rq}.csv", index=False)   

In [ ]:
# Load transformation registry 
with open("transform_registry.pkl", "rb") as f:
    registry = pickle.load(f)

#### Scenario 1: Infection Episode -> Outcome Measures

In [ ]:
# Define treatments and outcomes
treatments = ["infection_episode"]
outcomes = ["post_num_symp_prop", "post_symp_sev_prop", "post_symp_freq_prop"]

run_effect_and_interventions(G_best, scm_best, final_df_raw, final_df_std, registry, treatments, outcomes, rq=1, n_reps=100)

#### Scenario 2: Infection Episode -> Other Factors

In [ ]:
# Define treatments and outcomes
other_outcomes = []
for col in final_df_std.columns:
    if (col.startswith("acute") | col.startswith("post")) & (col not in outcomes):
        other_outcomes.append(col)

run_effect_and_interventions(G_best, scm_best, final_df_raw, final_df_std, registry, treatments, other_outcomes, rq=2, n_reps=100)

#### Scenario 3: Other Factors -> Outcome Measures

In [ ]:
# Define treatments and outcomes
other_treatments = []
for col in final_df_std.columns:
    if ((col.count("symp")==0) & ((col.startswith("acute") | col.startswith("pre")))) | (col in [
        "age", "is_female", "long_covid", "me_cfs", "dysautonomia", "fibromyalgia", "period_at_covid_start"]):
        other_treatments.append(col)

run_effect_and_interventions(G_best, scm_best, final_df_raw, final_df_std, registry, other_treatments, outcomes, rq=3, n_reps=100)

### Pathway analysis

In [ ]:
def pathway_analysis(G, treatment, outcome):
    
    # 1. Treatment Edge Analysis
    # Analyze the strength of the edges from the treatment
    print("\n1. Treatment Edge Analysis")
    edge_weights = nx.get_edge_attributes(G, 'weight')

    # Find edges FROM treatment
    treatment_edges = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges() if u == treatment]

    print(f"Direct effects of {treatment}:")
    for u, v, weight in sorted(treatment_edges, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {u} → {v}: {weight:.3f}")

    # 2. Outcome Edge Analysis
    # Analyze the strength of the edges to the outcome
    print("\n2. Outcome Edge Analysis")

    # Find edges INTO outcome
    outcome_edges = [(u, v, edge_weights.get((u,v), 0)) 
                    for u, v in G.edges() if v == outcome]

    print(f"Direct edges into {outcome}:")
    for u, v, weight in sorted(outcome_edges, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {u} → {v}: {weight:.3f}")

In [ ]:
# Check for all paths from treatment to outcome
treatment = 'infection_episode'
outcome = 'post_symp_sev_prop'

pathway_analysis(G_best, treatment, outcome)

In [ ]:
# Check for all paths from treatment to outcome
treatment = 'infection_episode'
outcome = 'post_symp_freq_prop'

pathway_analysis(G_best, treatment, outcome)

In [ ]:
# Check for all paths from treatment to outcome
treatment = 'infection_episode'
outcome = 'acute_resting_hr'

pathway_analysis(G_best, treatment, outcome)

In [ ]:
# Check for all paths from treatment to outcome
treatment = 'pre_sleep'
outcome = 'post_symp_sev_prop'

pathway_analysis(G_best, treatment, outcome)

# Check for all paths from treatment to outcome
treatment = 'acute_sleep'
outcome = 'post_symp_sev_prop'

pathway_analysis(G_best, treatment, outcome)

# Check for all paths from treatment to outcome
treatment = 'pre_emotionally_stressful'
outcome = 'post_symp_sev_prop'

pathway_analysis(G_best, treatment, outcome)